In [1]:
from typing import Dict
from pydantic_ai import Agent
from eicat_ai import agents
from eicat_ai.models import Paper
from eicat_ai.converters import pdf_to_markdown

First define a list of potential models to test the agent with.

In [2]:
models: Dict[str, str] = {
    "nova-lite": "bedrock:amazon.nova-lite-v1:0",
	"nova-pro": "bedrock:amazon.nova-pro-v1:0",
    "claude": "bedrock:anthropic.claude-3-7-sonnet-20250219-v1:0",
}

Next create the agent with your selected model.

In [3]:
paper_agent: Agent[None, Paper] = agents.paper_agent(models["claude"])

Use the agent to convert a PDF into the expected pydantic model output.

In [4]:
paper: Paper = await pdf_to_markdown(paper_agent, "../data/Full references_Assignment-20251020/Gozlan et al. 2005.pdf")

Save the output to JSON so it can be re-read late.

In [ ]:
paper.save("../data/Gozlan.json")

Reload the paper if it has already been saved.

In [3]:
p: Paper = Paper.load("../data/Gozlan.json")

Create an agent for extracting EICAT impacts from the paper.

In [4]:
from eicat_ai.agents import data_extraction_agent
from eicat_ai.models import Impact
from typing import List

de: Agent[Paper, List[Impact]] = data_extraction_agent(models["claude"])

Run the agent requesting it extract the all impacts and providing the extracted paper as a dependency.

In [5]:
result = await de.run(["Extract all impacts for me"], deps=p)

Save the resulting output to csv.

In [6]:
Impact.save_to_csv(result.output, "../data/Gozlan_impacts.csv")